# CARE training notebook

This notebook trains a CARE model from a saved patch `.npz` file.

Workflow:
1. import training helpers from the Python module  
2. define user settings  
3. resolve the patch file and patch metadata  
4. load training / validation data  
5. (optional) normalize patches *(usually disabled)*  
6. build model config  
7. create model  
8. train  
9. save history + metadata  
10. inspect validation predictions  

> **Note:**  
> Training is typically performed on **raw patch values**.  
> Normalization is optional and should only be enabled if it matches how the data was prepared.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from ISS_CARE.ISS_CARE_training import (
    # Patch / data loading
    resolve_patch_file,
    resolve_patch_metadata_file,
    load_json_file,
    load_care_training_data,

    # Optional preprocessing
    normalize_patch_dataset,

    # Visualization
    plot_patch_examples,
    plot_training_curves,
    predict_on_validation_examples,

    # Model setup
    build_model_run_name,
    build_care_config,
    create_care_model,
    print_model_save_info,

    # Training
    train_care_model,
    save_training_history,

    # Metadata
    build_training_metadata,
    save_training_metadata,
)

## User settings

Set the patch file, training hyperparameters, and optional normalization behavior here.

Notes:
- `MODEL_NAME` is auto-generated later from the patch file name + timestamp.
- Training is typically performed on **raw patch values**; normalization is optional and should only be enabled if it matches how the data was prepared.
- `TRAIN_LOSS="mae"` is a common and robust default for CARE denoising.
- `UNET_N_DEPTH=2` is a good default for moderate patch sizes (e.g. 128×128).

In [ ]:
# ----------------------------
# User settings
# ----------------------------

HOME = Path.home()
CARE_ROOT = HOME / "moldia-archive" / "CARE_training"
CARE_ROOT = CARE_ROOT.expanduser().resolve()

# Patch file to train on
PATCH_FILE_NAME = "NON_DAPI_train_patches_Leica_all_01__2026-04-30_17-37.npz"
# PATCH_FILE_NAME = "DAPI_ONLY_train_patches_Leica40X_final.npz"

# Model output
MODEL_DIRNAME = "care_models"
MODEL_DIR = CARE_ROOT / MODEL_DIRNAME

# MODEL_NAME is built automatically later from patch file + timestamp
MODEL_NAME = None

# Validation split
VALIDATION_SPLIT = 0.1

# ----------------------------
# Training parameters
# ----------------------------
TRAIN_BATCH_SIZE = 8
TRAIN_STEPS_PER_EPOCH = 50
TRAIN_EPOCHS = 150

# ----------------------------
# Model parameters
# ----------------------------
UNET_KERN_SIZE = 3
UNET_N_DEPTH = 2
TRAIN_LEARNING_RATE = 2e-4
TRAIN_LOSS = "mae"
PROBABILISTIC = False

# ----------------------------
# Optional normalization (usually OFF)
# ----------------------------
NORMALIZE_PATCHES = False
NORMALIZATION_PMIN = 1.0
NORMALIZATION_PMAX = 99.8
NORMALIZATION_EPS = 1e-8


# ----------------------------
# Clear summary print
# ----------------------------
print("=" * 60)
print("CARE training setup")
print("=" * 60)

print(f"[PATH] CARE_ROOT: {CARE_ROOT}")
print(f"[PATH] PATCH_FILE_NAME: {PATCH_FILE_NAME}")
print(f"[PATH] MODEL_DIR: {MODEL_DIR}")

# Infer variant from filename
patch_name_upper = PATCH_FILE_NAME.upper()
if "NON_DAPI" in patch_name_upper:
    variant = "NON_DAPI (main model)"
elif "DAPI" in patch_name_upper:
    variant = "DAPI_ONLY (separate model)"
else:
    variant = "UNKNOWN"

print(f"[DATA] Training variant: {variant}")

print("\n[DATA] Validation split:", VALIDATION_SPLIT)

print("\n[TRAINING]")
print(f"  Batch size: {TRAIN_BATCH_SIZE}")
print(f"  Steps/epoch: {TRAIN_STEPS_PER_EPOCH}")
print(f"  Epochs: {TRAIN_EPOCHS}")

print("\n[MODEL]")
print(f"  UNet depth: {UNET_N_DEPTH}")
print(f"  Kernel size: {UNET_KERN_SIZE}")
print(f"  Learning rate: {TRAIN_LEARNING_RATE}")
print(f"  Loss: {TRAIN_LOSS}")
print(f"  Probabilistic: {PROBABILISTIC}")

print("\n[NORMALIZATION]")
print(f"  Enabled: {NORMALIZE_PATCHES}")
if NORMALIZE_PATCHES:
    print(f"  pmin/pmax: {NORMALIZATION_PMIN} / {NORMALIZATION_PMAX}")
    print(f"  eps: {NORMALIZATION_EPS}")
else:
    print("  Using raw patch values")

print("=" * 60)

## Resolve patch file and patch metadata

This step:

- resolves the selected patch `.npz` file  
- looks for its sidecar metadata file (`.json`, same filename stem)  
- prints a summary of the dataset (variant, shapes, patch counts)  
- automatically builds a unique model run name from the patch file + timestamp  

> The metadata file is optional but recommended.  
> It provides useful information about how the patches were generated (e.g. number of patches, axes, filtering).

In [ ]:
patch_dir, patch_file = resolve_patch_file(
    care_root=CARE_ROOT,
    patch_file_name=PATCH_FILE_NAME,
)

MODEL_NAME = build_model_run_name(
    patch_file=patch_file,
    prefix="CARE",
)

patch_metadata_file = resolve_patch_metadata_file(patch_file)
patch_metadata = load_json_file(patch_metadata_file) if patch_metadata_file else None

print("Resolved patch directory:", patch_dir)
print("Resolved patch file:", patch_file)
print("Patch metadata file:", patch_metadata_file)
print("Auto-generated MODEL_NAME:", MODEL_NAME)

if patch_metadata is not None:
    print("\nPatch metadata summary:")
    print("  variant:", patch_metadata.get("variant_name"))
    print("  axes:", patch_metadata.get("axes"))
    print("  X shape:", patch_metadata.get("X_shape"))
    print("  Y shape:", patch_metadata.get("Y_shape"))
    print("  pairs used:", patch_metadata.get("n_pairs_used"))
    print("  total patches:", patch_metadata.get("n_patches_total"))

    patch_params = patch_metadata.get("patch_parameters", {})
    if patch_params:
        print("\nPatch parameters:")
        for k, v in patch_params.items():
            print(f"  {k}: {v}")
else:
    print("\nPatch metadata file not found.")

## Load CARE training data

This step:

- loads training patches (`X`, `Y`)  
- splits off validation patches (`X_val`, `Y_val`)  
- returns axis information and channel counts  
- prints dataset statistics and sanity checks  

Sanity checks include:
- shapes of all arrays  
- min / max intensity values  
- detection of NaN / Inf values  
- detection of constant or near-constant patches  

> This is an important verification step before training.  
> If issues are detected here, they usually originate from patch generation.

In [ ]:
X, Y, X_val, Y_val, axes, n_channel_in, n_channel_out = load_care_training_data(
    patch_file=patch_file,
    validation_split=VALIDATION_SPLIT,
)



## Preview loaded patches

This step:

- displays a few example training patch pairs (`X` → `Y`)  
- shows input (top) and target (bottom) images  

> This is a quick visual sanity check to confirm:
> - source and target are correctly aligned  
> - patches contain meaningful signal  
> - no obvious artifacts are present  

**Note:**  
Images are **normalized for visualization only** (per patch pair) to improve contrast.  
This does **not reflect the actual values used for training**.

In [ ]:
plot_patch_examples(
    X,
    Y,
    n_show=5,
    title="Training patches (top: input, bottom: target; normalized for display)",
)

## Optional normalization

This step optionally applies **percentile normalization** to training and validation patches.

- Disabled by default (recommended)  
- Enable only if it matches how your data should be handled  

> In most cases, training on **raw patch values** works well and is preferred.

**Important:**
- This changes the **actual values used for training** (unlike the preview, which only normalizes for display)  
- If you train with normalization enabled,  
  you must apply the **same normalization during inference** for consistent results.

In [ ]:
if NORMALIZE_PATCHES:
    print("\n" + "=" * 60)
    print("[INFO] Applying percentile normalization to patches")
    print("=" * 60)
    print(f"pmin = {NORMALIZATION_PMIN}")
    print(f"pmax = {NORMALIZATION_PMAX}")
    print(f"eps  = {NORMALIZATION_EPS}")

    X, Y, X_val, Y_val = normalize_patch_dataset(
        X,
        Y,
        X_val,
        Y_val,
        enabled=True,
        pmin=NORMALIZATION_PMIN,
        pmax=NORMALIZATION_PMAX,
        eps=NORMALIZATION_EPS,
    )
else:
    print("[INFO] Skipping normalization")
    print("[INFO] Using raw patch values for training")


## Build CARE configuration

This step creates the CARE model configuration using:

- patch axes  
- input and output channel counts  
- training parameters  
- model architecture settings  

The configuration controls how the CARE network is built and trained.

In [ ]:
config = build_care_config(
    axes=axes,
    n_channel_in=n_channel_in,
    n_channel_out=n_channel_out,

    # Training parameters
    train_batch_size=TRAIN_BATCH_SIZE,
    train_steps_per_epoch=TRAIN_STEPS_PER_EPOCH,
    train_epochs=TRAIN_EPOCHS,
    train_learning_rate=TRAIN_LEARNING_RATE,

    # Model architecture
    unet_kern_size=UNET_KERN_SIZE,
    unet_n_depth=UNET_N_DEPTH,

    # Behavior
    probabilistic=PROBABILISTIC,
    train_loss=TRAIN_LOSS,
)

## Create model

This step creates the CARE model and output folder.

The model name is automatically built from:

- patch file stem  
- timestamp  

The model folder will contain training outputs such as weights, config, history, and metadata.

In [ ]:
model = create_care_model(
    config=config,
    model_name=MODEL_NAME,
    model_dir=MODEL_DIR,
)

print_model_save_info(
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
)

model_path = (MODEL_DIR / MODEL_NAME).resolve()


## Inspect the model architecture

This step displays the CARE model architecture.

It shows:
- layer structure  
- number of parameters  
- input/output shapes  

Use this to verify that:
- the model matches your expected configuration  
- channel dimensions are correct  
- the network size is appropriate for your data

In [ ]:
# Optional: set Keras model name for nicer logs
try:
    model.keras_model._name = MODEL_NAME
except Exception:
    pass

model.keras_model.summary()

## Train the model

This step trains the CARE model using the loaded training and validation patches.

Runtime depends on:

- dataset size  
- patch size  
- GPU availability  
- batch size  
- steps per epoch  
- number of epochs  

Training outputs are saved automatically in the model output folder.

In [ ]:
history = train_care_model(
    model=model,
    X=X,
    Y=Y,
    X_val=X_val,
    Y_val=Y_val,
)

## Save training history

This step saves the training curves and metrics as a JSON file in the model output folder.

The file contains:
- training loss  
- validation loss  
- any additional tracked metrics  

This is useful for:
- later analysis  
- reproducibility  
- plotting training performance without rerunning training  

In [ ]:
history_file = save_training_history(
    history,
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
)

print(f"[INFO] Training history saved to: {history_file}")

## Plot training curves

This step visualizes the training history.

It shows:
- training loss  
- validation loss  
- additional metrics (if available)  

Use this to:
- monitor convergence  
- detect overfitting (training ↓ while validation ↑)  
- assess overall training stability  

In [ ]:
plot_training_curves(history)

## Quick validation prediction check

This step runs the trained model on a few validation patches.

It shows:
- top: input  
- middle: target  
- bottom: prediction  

It also prints simple summary metrics:
- MAE (mean absolute error)  
- MSE (mean squared error)  

Use this to quickly verify that:
- predictions are reasonable  
- the model has learned the mapping  
- no obvious artifacts or failures are present  

In [ ]:
X_example, Y_example, Y_pred = predict_on_validation_examples(
    model=model,
    X_val=X_val,
    Y_val=Y_val,
    probabilistic=PROBABILISTIC,
    n_examples=10,
)

## Save training metadata

This step saves a JSON file with all relevant information about the training run.

It includes:
- model hyperparameters  
- patch file used  
- patch metadata (if available)  
- data shapes (train/validation)  
- normalization settings  

This ensures:
- full reproducibility  
- traceability of how the model was trained  
- easier debugging and comparison between runs  

In [ ]:
metadata = build_training_metadata(
    care_root=CARE_ROOT,
    patch_dir=patch_dir,
    patch_file=patch_file,
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,

    # Training parameters
    validation_split=VALIDATION_SPLIT,
    train_batch_size=TRAIN_BATCH_SIZE,
    train_steps_per_epoch=TRAIN_STEPS_PER_EPOCH,
    train_epochs=TRAIN_EPOCHS,
    train_learning_rate=TRAIN_LEARNING_RATE,
    train_loss=TRAIN_LOSS,
    probabilistic=PROBABILISTIC,

    # Model architecture
    unet_kern_size=UNET_KERN_SIZE,
    unet_n_depth=UNET_N_DEPTH,

    # Data
    axes=axes,
    X_shape=X.shape,
    Y_shape=Y.shape,
    X_val_shape=X_val.shape,
    Y_val_shape=Y_val.shape,
    n_channel_in=n_channel_in,
    n_channel_out=n_channel_out,

    # Normalization
    normalization_enabled=NORMALIZE_PATCHES,
    normalization_pmin=NORMALIZATION_PMIN,
    normalization_pmax=NORMALIZATION_PMAX,
    normalization_eps=NORMALIZATION_EPS,
)

metadata_file = save_training_metadata(
    metadata,
    model_dir=MODEL_DIR,
    model_name=MODEL_NAME,
)

print(f"[INFO] Training metadata saved to: {metadata_file}")

## Final output summary

This final step summarizes the key outputs of the training run.

It reports:
- patch file used for training  
- patch metadata file (if available)  
- model output directory  
- training history file  
- training metadata file  

Use this as a quick reference to locate all relevant files generated during training.

In [ ]:
print("Training complete.")
print("Patch file used:", patch_file)
print("Patch metadata file:", patch_metadata_file)
print("Model output directory:", (MODEL_DIR / MODEL_NAME).resolve())
print("Training history file:", history_file)
print("Training metadata file:", metadata_file)

## Load the model later

To load the trained model in another notebook:

```python
from csbdeep.models import CARE

model = CARE(
    config=None,  # config is loaded automatically from config.json
    name=MODEL_NAME,
    basedir=str(MODEL_DIR),
)